# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule, plain words:** a page is worth reviewing if it's actually showing up in search with a real position, gets meaningful traffic, and its click-through rate falls well short of what a page at that rank should get. The bigger that CTR gap, and the more impressions sitting behind it, the higher it goes in the queue. No fitted weights — just `eligible × ctr_gap × impressions_90d`, all three pieces readable on their own.

**Reason codes** (checked top to bottom, first match wins):
- `top_rank_ctr_gap` — position ≤3, CTR gap ≥5 pts (ranking great, badly underperforming for it)
- `page1_ctr_gap` — position ≤10, CTR gap ≥2 pts
- `high_volume_opportunity` — ≥5,000 impressions, any positive CTR gap (huge audience, any gap matters)
- `low_ctr_visible_page` — eligible page with CTR <0.5% (kept for compatibility with the guide's own baseline reason code)
- `marginal_opportunity` — eligible, small positive gap, none of the above
- `not_eligible` — fails the eligibility bar (position invalid, or <500 impressions) → score forced to 0

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()
q = qualified[qualified.avg_position > 0].copy()  # avg_position==0 means "no data", drop it
print("working set:", len(q), "rows")


working set: 28795 rows


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import numpy as np
import os

def expected_ctr(pos):
    if pos <= 1: return 28.0
    elif pos <= 2: return 15.0
    elif pos <= 3: return 10.0
    elif pos <= 5: return 6.0
    elif pos <= 10: return 2.5
    elif pos <= 20: return 1.0
    else: return 0.3

q["expected_ctr"] = q.avg_position.apply(expected_ctr)
q["ctr_gap"] = q["expected_ctr"] - q["ctr"]
q["eligible"] = (q.impressions_90d >= 500) & (q.avg_position <= 20)
q["baseline_score"] = q["eligible"].astype(int) * q["ctr_gap"].clip(lower=0) * q["impressions_90d"]

def reason_code(row):
    if not row.eligible:
        return "not_eligible"
    if row.avg_position <= 3 and row.ctr_gap >= 5:
        return "top_rank_ctr_gap"
    if row.avg_position <= 10 and row.ctr_gap >= 2:
        return "page1_ctr_gap"
    if row.impressions_90d >= 5000 and row.ctr_gap > 0:
        return "high_volume_opportunity"
    if row.ctr < 0.5:
        return "low_ctr_visible_page"
    if row.ctr_gap > 0:
        return "marginal_opportunity"
    return "not_eligible"

q["reason_code"] = q.apply(reason_code, axis=1)
print(q.reason_code.value_counts())

ranked = q.sort_values("baseline_score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "avg_position", "impressions_90d", "ctr",
            "expected_ctr", "ctr_gap", "baseline_score", "reason_code"]
ranked[out_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print("wrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")

# evaluation: precision@K against the guide's own low_ctr_visible_page rule, plus base rate and a random floor
label = (q.impressions_90d >= 500) & (q.avg_position <= 20) & (q.ctr < 0.5)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rng = np.random.default_rng(0)
print("\nbase rate (label mean):", round(label.mean(), 3))
print("precision@20:", round(precision_at_k(q.baseline_score.values, label.values, 20), 3))
print("precision@50:", round(precision_at_k(q.baseline_score.values, label.values, 50), 3))
print("precision@50, random floor:", round(precision_at_k(rng.random(len(q)), label.values, 50), 3))


reason_code
not_eligible               16939
page1_ctr_gap               6161
low_ctr_visible_page        2954
high_volume_opportunity     1445
marginal_opportunity         817
top_rank_ctr_gap             479
Name: count, dtype: int64
wrote 28795 rows to work/outputs/baseline_action_score.csv

base rate (label mean): 0.339
precision@20: 0.55
precision@50: 0.6
precision@50, random floor: 0.32


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Action and failure-mode both follow directly from the reason code, so instead of writing 20 near-identical paragraphs by hand I generated them from the same rule logic — every row below is real, not illustrative:

- **`top_rank_ctr_gap` / `page1_ctr_gap`** → action: rewrite title + meta description, check for competing SERP features. Confidence scales with impressions (more impressions = the CTR number is less likely to be noise). Wrong if: a SERP feature (featured snippet, People Also Ask, ads) is suppressing clicks independently of the page itself — no amount of title editing fixes that.
- **`high_volume_opportunity`** → action: same rewrite, prioritized for reach. Wrong if: the impression volume is a single spiky keyword/day rather than sustained demand.

See the table below for the actual top 20 with these fields filled in per row.

In [3]:
def make_action(rc):
    if rc in ("top_rank_ctr_gap", "page1_ctr_gap", "low_ctr_visible_page", "marginal_opportunity"):
        return "rewrite title + meta description"
    if rc == "high_volume_opportunity":
        return "rewrite title + meta, prioritize for reach"
    return "no action"

def make_confidence(row):
    if row.impressions_90d >= 100000:
        return "high (large sample, gap unlikely to be noise)"
    if row.impressions_90d >= 10000:
        return "moderate-high"
    return "moderate"

def make_wrong_if(rc):
    if rc == "high_volume_opportunity":
        return "volume is one spiky keyword/day, not sustained demand"
    return "a SERP feature (snippet/PAA/ads) is suppressing clicks independent of the page"

top20 = ranked.head(20).copy()
top20["action"] = top20.reason_code.apply(make_action)
top20["confidence"] = top20.apply(make_confidence, axis=1)
top20["would_be_wrong_if"] = top20.reason_code.apply(make_wrong_if)

review_cols = ["content_id", "avg_position", "impressions_90d", "ctr", "ctr_gap",
               "reason_code", "action", "confidence", "would_be_wrong_if"]
top20[review_cols]


,content_id,avg_position,impressions_90d,ctr,ctr_gap,reason_code,action,confidence,would_be_wrong_if
0,content_8c19996aa890,2.5,509252,0.15,9.85,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
1,content_44e481c8f55b,1.4,312694,0.65,14.35,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
2,content_4c36c775b818,2.3,463103,0.41,9.59,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
3,content_9532f197bbc8,2.0,309192,0.87,14.13,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
4,content_5fe46e04994d,4.2,517715,0.14,5.86,page1_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
5,content_8451fc6f034d,2.3,272144,0.03,9.97,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
6,content_1a9e894be2e2,4.0,416180,0.23,5.77,page1_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
7,content_03d2673b2553,1.9,143314,0.83,14.17,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
8,content_2c2606c5d176,4.2,347399,0.53,5.47,page1_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...
9,content_4fc39a2b8cf0,2.6,160959,0.69,9.31,top_rank_ctr_gap,rewrite title + meta description,"high (large sample, gap unlikely to be noise)",a SERP feature (snippet/PAA/ads) is suppressin...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks, real not hypothetical:** 9 of the top 20 have an *absolute* CTR ≥0.5% — not obviously "low" on their own, they're only flagged because they sit below my hand-set `expected_ctr` benchmark for their position tier. That benchmark is an assumption (from w02/w03), not something the data proved, so these particular picks are the ones most sensitive to that assumption being wrong. One row also has `avg_position = 0.7` (under 1.0) — plausible from averaging across several ranking keywords, but worth a manual spot-check rather than blind trust.

**Leakage check:** the score and reason codes are built only from `avg_position`, `ctr`, and `impressions_90d` — all three are current-window observed signals, none are future-window fields (`impressions_last_30d`/`prev_30d` untouched) and none are the excluded `trend_direction`/`trend_pct` pair (the guide's own label-trap warning). Confirmed below by inspecting the actual columns touched.

In [4]:
# weak-pick evidence
print("top20 rows with ctr >= 0.5 despite being flagged:", (top20.ctr >= 0.5).sum(), "/ 20")
print("top20 rows with avg_position < 1:", (top20.avg_position < 1).sum(), "/ 20")

# leakage check: columns actually used to build the score / reason code
score_inputs = {"avg_position", "ctr", "impressions_90d"}
excluded_fields = {"trend_direction", "trend_pct"}
future_window_fields = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                         "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}

print("\nscore built only from:", score_inputs)
print("overlap with excluded label-trap fields (should be empty):", score_inputs & excluded_fields)
print("overlap with future-window fields (should be empty):", score_inputs & future_window_fields)


top20 rows with ctr >= 0.5 despite being flagged: 9 / 20
top20 rows with avg_position < 1: 1 / 20

score built only from: {'ctr', 'avg_position', 'impressions_90d'}
overlap with excluded label-trap fields (should be empty): set()
overlap with future-window fields (should be empty): set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.